# Calculating agent capacity

This notebook contains example data processing using the output of an example
model.

Output files are mostly in CSV format. The format of output files is documented [here][output-format].

We begin by loading the `model.toml` input file to get the list of milestone years.

[output-format]: https://energysystemsmodellinglab.github.io/MUSE2/file_formats/output_files.html

In [ ]:
import tomllib

import pandas as pd

from muse2_data_analysis.helpers import get_example_input_dir, get_example_output_dir

INPUT_DIR = get_example_input_dir()
OUTPUT_DIR = get_example_output_dir()

with (INPUT_DIR / "model.toml").open("rb") as f:
    model = tomllib.load(f)

# We need to know the milestone years for processing the assets file
years = model["milestone_years"]

## Load and process output data

We next load the output data. In this case, we want to calculate how much capacity was invested in
different processes for different agents. This information can be found in the `assets.csv` output
file.

The `assets.csv` file contains information about different assets, including when they were
commissioned and decommissioned as well as their capacity. To calculate the overall capacity for a
given agent and process type, we have to process this data. Note that different assets owned by the
same agent may have the same process ID if the agent has reinvested in the same process type in a
different year.

In [ ]:
# The assets.csv file contains info about which assets were invested in and when
assets = pd.read_csv(OUTPUT_DIR / "assets.csv")
# The asset_capacities.csv file contains info about the capacities for each asset
# along the simulation
asset_capacities = pd.read_csv(OUTPUT_DIR / "asset_capacities.csv")


# We define a helper function to bring some useful information from 'assets' into
#'assets_capacity'.
def get_agent_and_process(x: pd.Series) -> pd.Series:
    """Collects "agent_id", "process_id", "commission_year" from assets."""
    col, val = (
        ("asset_id", x.asset_id) if x.asset_id is not None else ("group_id", x.group_id)
    )
    row = assets[assets[col] == val].iloc[0]
    return row[["agent_id", "process_id", "commission_year"]]


asset_capacities = pd.concat(
    [asset_capacities, asset_capacities.apply(get_agent_and_process, axis=1)], axis=1
)

# Calculate capacity for each type of process for each agent
capacity = pd.DataFrame()
for year in years:
    active = asset_capacities[year >= asset_capacities["commission_year"]]

    # This only works because each agent is responsible for one and only one commodity
    cap_sum = active.groupby(["agent_id", "process_id"])["capacity"].sum().reset_index()

    df = pd.DataFrame(cap_sum)
    df["year"] = year

    capacity = pd.concat([capacity, df])

capacity

## Plot results

Finally, we plot the results.

Note that each of the agents has invested in only one process type; otherwise there would be
multiple bars per plot here.

In [ ]:
import matplotlib.pyplot as plt

agents = capacity["agent_id"].unique()
_, axes = plt.subplots(1, len(agents), figsize=(4 * len(agents), 4))
for ax, agent in zip(axes, agents):
    capacity[capacity["agent_id"] == agent].pivot(
        index="year", columns="process_id", values="capacity"
    ).plot(kind="bar", ax=ax)
    ax.set_title(agent)
    ax.set_xlabel("Year")
    ax.set_ylabel("Capacity")
    ax.legend(
        title="Process", bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0
    )

plt.tight_layout()